# Advanced Problems: Variable Equality

Practice identity, equality, aliasing, mutation, `None`, numeric equality, and custom equality behavior.

Each problem includes a solution after the prompt.

## Problem 1 — Identity vs Equality Prediction

Predict the output before running the cell.

For each pair, decide whether `is` and `==` return `True` or `False`.

In [1]:
pairs = [
    (10, 10),
    (10, 10.0),
    (10, 10 + 0j),
    ([1, 2], [1, 2]),
    ((1, 2), (1, 2)),
    (None, None),
]

for left, right in pairs:
    print(f"{left!r:>10} and {right!r:<10} | is: {left is right:<5} | ==: {left == right}")

        10 and 10         | is: 1     | ==: True
        10 and 10.0       | is: 0     | ==: True
        10 and (10+0j)    | is: 0     | ==: True
    [1, 2] and [1, 2]     | is: 0     | ==: True
    (1, 2) and (1, 2)     | is: 1     | ==: True
      None and None       | is: 1     | ==: True


### Solution 1

`is` checks whether two variables refer to the same object.

`==` checks whether two objects compare as equal by value.

Important observations:

- `10 == 10.0 == 10 + 0j` is `True`, even though these objects have different types.
- Two separately created lists with the same contents compare equal, but are not the same object.
- `None` is a singleton, so `None is None` is `True`.
- Some immutable literals may share references as an implementation detail, but code should not rely on that except for singletons such as `None`.

## Problem 2 — Aliasing and Mutation

Without running the code first, predict the final values of `a`, `b`, `a is b`, and `a == b`.

In [2]:
a = [1, 2, 3]
b = a

b.append(4)
a[0] = 99

print("a:", a)
print("b:", b)
print("a is b:", a is b)
print("a == b:", a == b)

a: [99, 2, 3, 4]
b: [99, 2, 3, 4]
a is b: True
a == b: True


### Solution 2

`b = a` does not create a new list. It creates a second reference to the same list.

Therefore, mutations through either name affect the same underlying object.

Expected result:

```python
a: [99, 2, 3, 4]
b: [99, 2, 3, 4]
a is b: True
a == b: True
```

## Problem 3 — Shallow Copy Trap

The following code creates a shallow copy.

Predict which comparisons are `True` and which are `False`.

In [3]:
a = [[1, 2], [3, 4]]
b = a.copy()

print("a is b:", a is b)
print("a == b:", a == b)
print("a[0] is b[0]:", a[0] is b[0])

b[0].append(999)

print("a:", a)
print("b:", b)
print("a == b:", a == b)

a is b: False
a == b: True
a[0] is b[0]: True
a: [[1, 2, 999], [3, 4]]
b: [[1, 2, 999], [3, 4]]
a == b: True


### Solution 3

`a.copy()` creates a new outer list, so `a is b` is `False`.

However, the inner lists are still shared. Therefore, `a[0] is b[0]` is `True`.

After `b[0].append(999)`, both `a` and `b` show the mutation because they share the same inner list.

This is the key difference between a shallow copy and a deep copy.

## Problem 4 — Fix the Copying Bug

The function below is intended to return a modified copy of a nested list without changing the original.

Fix it.

In [4]:
def add_bonus_bad(records):
    copied = records.copy()
    copied[0].append("BONUS")
    return copied

original = [["Alice", 90], ["Bob", 85]]
result = add_bonus_bad(original)

print("original:", original)
print("result:", result)

original: [['Alice', 90, 'BONUS'], ['Bob', 85]]
result: [['Alice', 90, 'BONUS'], ['Bob', 85]]


### Solution 4

In [5]:
from copy import deepcopy

def add_bonus(records):
    copied = deepcopy(records)
    copied[0].append("BONUS")
    return copied

original = [["Alice", 90], ["Bob", 85]]
result = add_bonus(original)

print("original:", original)
print("result:", result)
print("original is result:", original is result)
print("original[0] is result[0]:", original[0] is result[0])

original: [['Alice', 90], ['Bob', 85]]
result: [['Alice', 90, 'BONUS'], ['Bob', 85]]
original is result: False
original[0] is result[0]: False


`deepcopy` recursively copies nested mutable objects.

Now the outer list and the inner lists are distinct objects.

## Problem 5 — Correctly Testing for `None`

The following function has a bug-prone comparison.

Rewrite it using best practice.

In [6]:
def describe_value_bad(value):
    if value == None:
        return "missing"
    return f"value is {value!r}"

print(describe_value_bad(None))
print(describe_value_bad(0))
print(describe_value_bad([]))

missing
value is 0
value is []


### Solution 5

In [7]:
def describe_value(value):
    if value is None:
        return "missing"
    return f"value is {value!r}"

print(describe_value(None))
print(describe_value(0))
print(describe_value([]))

missing
value is 0
value is []


Use `is None`, not `== None`.

`None` is a singleton, so identity comparison is the intended and safest test.

## Problem 6 — Dangerous Custom Equality

A class can define its own equality behavior.

Predict the output.

In [8]:
class AlwaysEqual:
    def __eq__(self, other):
        return True

x = AlwaysEqual()
y = AlwaysEqual()

print("x is y:", x is y)
print("x == y:", x == y)
print("x == None:", x == None)
print("x is None:", x is None)

x is y: False
x == y: True
x == None: True
x is None: False


### Solution 6

`x is y` is `False` because `x` and `y` are different objects.

`x == y` is `True` because `AlwaysEqual.__eq__` always returns `True`.

`x == None` is also `True`, which shows why `== None` can be unsafe.

`x is None` is `False`, because `x` is not the singleton `None` object.

## Problem 7 — Implement Value Equality

Complete the `Point` class so two points compare equal when their coordinates are equal.

Also verify that different instances are not identical.

In [9]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        # TODO: implement this
        pass

p1 = Point(2, 3)
p2 = Point(2, 3)
p3 = Point(3, 2)

print(p1 is p2)
print(p1 == p2)
print(p1 == p3)

False
None
None


### Solution 7

In [10]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

p1 = Point(2, 3)
p2 = Point(2, 3)
p3 = Point(3, 2)

print("p1 is p2:", p1 is p2)
print("p1 == p2:", p1 == p2)
print("p1 == p3:", p1 == p3)
print("p1 == (2, 3):", p1 == (2, 3))

p1 is p2: False
p1 == p2: True
p1 == p3: False
p1 == (2, 3): False


Best practice: return `NotImplemented` when comparing against an unsupported type.

That gives the other object a chance to handle the comparison.

## Problem 8 — Numeric Equality and Dictionary Keys

Predict the length and contents of the dictionary.

In [11]:
d = {}
d[10] = "integer"
d[10.0] = "float"
d[10 + 0j] = "complex"

print(d)
print(len(d))
print(10 == 10.0 == 10 + 0j)
print(hash(10), hash(10.0), hash(10 + 0j))

{10: 'complex'}
1
True
10 10 10


### Solution 8

The dictionary has length `1`.

`10`, `10.0`, and `10 + 0j` compare equal and have compatible hashes, so they represent the same dictionary key.

The final assignment overwrites the previous value.

Expected final value:

```python
{10: 'complex'}
```

## Problem 9 — Find the Bug: Shared Default List

This function accidentally reuses the same list across calls.

Explain the bug and fix it.

In [12]:
def append_item_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

first = append_item_bad("a")
second = append_item_bad("b")

print(first)
print(second)
print(first is second)

['a', 'b']
['a', 'b']
True


### Solution 9

In [13]:
def append_item(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

first = append_item("a")
second = append_item("b")

print(first)
print(second)
print(first is second)

['a']
['b']
False


Default argument values are created once when the function is defined, not each time the function is called.

Using `None` as the default sentinel avoids accidentally sharing a mutable object.

## Problem 10 — Identity-Based Sentinel

Create a unique sentinel object called `MISSING`.

Use it to distinguish between:

- no argument provided
- argument explicitly provided as `None`
- argument provided as another value

In [14]:
# TODO: create a sentinel object and complete the function

def classify(value):
    pass

print(classify())
print(classify(None))
print(classify(0))

TypeError: classify() missing 1 required positional argument: 'value'

### Solution 10

In [15]:
MISSING = object()

def classify(value=MISSING):
    if value is MISSING:
        return "no argument provided"
    if value is None:
        return "explicit None provided"
    return f"value provided: {value!r}"

print(classify())
print(classify(None))
print(classify(0))

no argument provided
explicit None provided
value provided: 0


`object()` creates a unique object.

Because no other object can accidentally be that exact object, identity comparison with `is` is appropriate here.